# RF–EBB tuning-curve pair comparison

Same calculation and panel functions as `tc_comparison_pairs.ipynb`, with RF and EBB as the two modalities. HD class 3 defines eligibility; **no ADN restriction**. One paired recording per mouse in the current poster cohort. M14/M15 have no comparable saved rectangular EBB bundle.

RF settings match that notebook: saved excitatory RF-row projection, at most two zero native bins, 30 comparison bins and 15 histogram bins. EBB curves are finite-cell means of saved smoothed rate maps, with unvisited cells masked. All range means the complete **saved** distance range (reported below), not extrapolation beyond it. Separate analyses cover all distances, 0–8, 8–16 and >16 cm. Distance bands follow native bin centers.

Same-unit peak scatter, native/aligned/sum heatmaps, peak-sum histogram and pairwise peak-distance density use the existing implementation. Mantel tests use unique off-diagonal cell pairs and shuffle whole unit identities; pooled permutations stay within recording. Ordered-pair density panels retain the original notebook's display convention; exported pair tables and overview use each off-diagonal pair once. Pooled results are descriptive cell-level evidence, not an animal-level population test.

In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from Utils.direction_comparison import concat
from Utils.tc_comparison import calculate_tc_statistics, plot_tc_comparison
from Utils.plotting import LIGHT_PLOT_STYLE
from scripts.rf_ebb_comparison import (load_pair_inputs, prepare_pair_inputs, save_pair_result,
                                      source_manifest, pairwise_overview, BAND_LABELS)
%matplotlib inline
root = Path('/mnt/senzailab/Kai/#Recording')
output_dir = Path(os.environ.get('RF_EBB_OUTPUT_DIR', 'rf_ebb_comparison_output/current'))
output_dir.mkdir(parents=True, exist_ok=True)
rf_only, rf_max_zero_bins = True, 2
comparison_bins, statistics_bins = 30, 15
n_permutations, random_seed = 10000, 1
plot_figsize = (8, 8)
plt.rcParams.update({**LIGHT_PLOT_STYLE, 'figure.facecolor':'white', 'axes.facecolor':'white',
                    'savefig.facecolor':'white', 'savefig.transparent':False, 'svg.fonttype':'none'})
recordings = [
    dict(mouse='m19', date='260827', probe='A', hd=9, rf=2, ebb=12),
    dict(mouse='m21', date='261006', probe='A', hd=9, rf=3, ebb=11),
    dict(mouse='m22', date='261008', probe='A', hd=9, rf=2, ebb=11),
    dict(mouse='m23', date='261015', probe='A', hd=9, rf=2, ebb=11),
    dict(mouse='m24', date='261104', probe='A', hd=10, rf=2, ebb=11),
]
prepared, manifests, summary, statistics_by_recording = {}, [], [], {}
def compare_pair(name, band, rf, ebb, *, groups=None, show=False):
    stats = calculate_tc_statistics(rf, ebb, bins=statistics_bins, is_wrap=True,
        n_permutations=n_permutations, random_seed=random_seed, permutation_groups=groups)
    figures = plot_tc_comparison(rf, ebb, stats, figsize=plot_figsize, show=False)
    record = save_pair_result(output_dir, name, band, rf, ebb, stats, figures)
    for fig, _ in figures:
        if show:
            display(fig)
        plt.close(fig)
    summary.append(dict(recording=name, band=band, cells=record['cells'], unique_pairs=record['unique_pairs'],
        **(record['direction']['pairwise'] if record['direction'] else {})))
    return stats


## Load, select, verify identity

Every input is read from recordings. No previous poster output or cached statistics are inputs. Kilosort source identity must agree across RF, HD and EBB sessions. Selection failures remain visible in the exported audit.

In [ ]:
for spec in recordings:
    name = spec['mouse'] + '_' + spec['date']
    loaded = load_pair_inputs(root, spec, rf_only=rf_only)
    prepared[name] = prepare_pair_inputs(loaded, bins=comparison_bins, max_zero_bins=rf_max_zero_bins)
    manifests.append(source_manifest(loaded))
    prepared[name]['selection'].to_csv(output_dir / (name + '_selection.csv'), index=False)
    print(name, prepared[name]['counts'], 'distance support:', loaded['maps']['distance_edges'][[0,-1]])
(output_dir / 'sources.json').write_text(json.dumps(manifests, indent=2) + '\n')
display(pd.concat([p['selection'].assign(recording=name) for name,p in prepared.items()]).groupby(['recording','band']).selected.agg(['sum','count']))

## m19_260827 — RF vs EBB

In [ ]:
name = 'm19_260827'
statistics_by_recording[name] = {}
for band, (rf, ebb) in prepared[name]['pairs'].items():
    statistics_by_recording[name][band] = compare_pair(name, band, rf, ebb)
fig = pairwise_overview(statistics_by_recording[name], title=name.replace('_',' ') + ' · RF vs EBB')
for extension in ('svg','png'):
    fig.savefig(output_dir / f'{name}_pairwise.{extension}', dpi=180, facecolor='white', transparent=False)
display(fig)
plt.close(fig)

## m21_261006 — RF vs EBB

In [ ]:
name = 'm21_261006'
statistics_by_recording[name] = {}
for band, (rf, ebb) in prepared[name]['pairs'].items():
    statistics_by_recording[name][band] = compare_pair(name, band, rf, ebb)
fig = pairwise_overview(statistics_by_recording[name], title=name.replace('_',' ') + ' · RF vs EBB')
for extension in ('svg','png'):
    fig.savefig(output_dir / f'{name}_pairwise.{extension}', dpi=180, facecolor='white', transparent=False)
display(fig)
plt.close(fig)

## m22_261008 — RF vs EBB

In [ ]:
name = 'm22_261008'
statistics_by_recording[name] = {}
for band, (rf, ebb) in prepared[name]['pairs'].items():
    statistics_by_recording[name][band] = compare_pair(name, band, rf, ebb)
fig = pairwise_overview(statistics_by_recording[name], title=name.replace('_',' ') + ' · RF vs EBB')
for extension in ('svg','png'):
    fig.savefig(output_dir / f'{name}_pairwise.{extension}', dpi=180, facecolor='white', transparent=False)
display(fig)
plt.close(fig)

## m23_261015 — RF vs EBB

In [ ]:
name = 'm23_261015'
statistics_by_recording[name] = {}
for band, (rf, ebb) in prepared[name]['pairs'].items():
    statistics_by_recording[name][band] = compare_pair(name, band, rf, ebb)
fig = pairwise_overview(statistics_by_recording[name], title=name.replace('_',' ') + ' · RF vs EBB')
for extension in ('svg','png'):
    fig.savefig(output_dir / f'{name}_pairwise.{extension}', dpi=180, facecolor='white', transparent=False)
display(fig)
plt.close(fig)

## m24_261104 — RF vs EBB

In [ ]:
name = 'm24_261104'
statistics_by_recording[name] = {}
for band, (rf, ebb) in prepared[name]['pairs'].items():
    statistics_by_recording[name][band] = compare_pair(name, band, rf, ebb)
fig = pairwise_overview(statistics_by_recording[name], title=name.replace('_',' ') + ' · RF vs EBB')
for extension in ('svg','png'):
    fig.savefig(output_dir / f'{name}_pairwise.{extension}', dpi=180, facecolor='white', transparent=False)
display(fig)
plt.close(fig)

## Pooled paired cells — all five mice

Both curves retain full `(mouse,date,probe,unit)` identity. Full-range panels are displayed below; all panels for every distance band and mouse are exported. Permutation groups are recording identities.

In [ ]:
pooled_statistics = {}
for band in BAND_LABELS:
    rf = concat(*(p['pairs'][band][0] for p in prepared.values()), label='Five mice RF')
    ebb = concat(*(p['pairs'][band][1] for p in prepared.values()), label='Five mice EBB · ' + BAND_LABELS[band])
    groups = pd.Series([':'.join(key.split(':')[:3]) for key in rf.index], index=rf.index)
    pooled_statistics[band] = compare_pair('pooled', band, rf, ebb, groups=groups, show=band=='all')
fig = pairwise_overview(pooled_statistics, title='RF vs EBB · five mice · permutations within recording')
for extension in ('svg','png'):
    fig.savefig(output_dir / f'pooled_pairwise.{extension}', dpi=180, facecolor='white', transparent=False)
display(fig)
plt.close(fig)
summary_table = pd.DataFrame(summary)
summary_table.to_csv(output_dir / 'summary.csv', index=False)
display(summary_table)
(output_dir / 'settings.json').write_text(json.dumps(dict(recordings=recordings,
    rf_only=rf_only, rf_max_zero_bins=rf_max_zero_bins, comparison_bins=comparison_bins,
    statistics_bins=statistics_bins, n_permutations=n_permutations, seed=random_seed,
    hd_class=3, adn_filter=False, pooled_permutation='within recording',
    excluded_mice={'m14':'No saved comparable rectangular EBB bundle', 'm15':'No saved comparable rectangular EBB bundle'}), indent=2)+'\n')

## Poster session — M19 260831, RF S1 / EBB S2

The user-selected session is shown separately from the original five-mouse pooled analysis. This section uses the poster’s complete HD-class-3 + ON-RF cohort and its all-height 0–200 ms RF projection. No per-cell EBB stability filter is applied. Colors span zero to each cell’s maximum. Raw reconstruction, unit identities and temporal stability are checked by the same loader as the poster.


In [ ]:
from scripts.poster_rf_ebb import load_sorting_inputs, prepare_sorting_profiles
from Utils.direction_comparison import normalize_tc
poster_spec = dict(mouse='m19', date='260831', probe='A', hd=2, rf=1, ebb=2)
poster_loaded = load_sorting_inputs(root, poster_spec)
poster_prepared = prepare_sorting_profiles(poster_loaded)
poster_profiles = {key: normalize_tc(table) for key, table in poster_prepared['profiles'].items()}
poster_profiles['RF'].attrs['label'] = 'M19 260831 RF S1 · all-height projection'
poster_stats = {}
for band in BAND_LABELS:
    poster_profiles[band].attrs['label'] = 'M19 260831 EBB S2 · ' + BAND_LABELS[band]
    poster_stats[band] = compare_pair('m19_260831_poster', band, poster_profiles['RF'],
        poster_profiles[band], show=band == 'all')
fig = pairwise_overview(poster_stats, title='Poster session · M19 260831 · RF S1 / EBB S2')
for extension in ('svg','png'):
    fig.savefig(output_dir / f'm19_260831_poster_pairwise.{extension}', dpi=180,
        facecolor='white', transparent=False)
display(fig)
plt.close(fig)
poster_prepared['metrics'].to_csv(output_dir / 'm19_260831_poster_quality.csv')
poster_manifest = dict(**source_manifest(poster_loaded), cohort=poster_prepared['metadata'],
    included_in_five_mouse_pool=False)
(output_dir / 'm19_260831_poster_sources.json').write_text(json.dumps(poster_manifest, indent=2) + '\n')
display(poster_prepared['metrics'][['half_r', 'unheld_r', 'mean_hz']])
